# Skin EPA 라벨 조합 실험
A: 1·2=1, 3·4=0. B: 1·2·3=1, 4=0. 공통 구조 특성 2265개, 원 EPA 등급 기반 그룹 외부5/내부3 교차검증, CatBoost를 제외한 6개 모델·각 20개 설정. 전처리와 가중치 및 임계값 선택은 학습 fold 안에서만 수행합니다.

In [1]:
import os, sys
from pathlib import Path
OUTPUT_ROOT = Path(r'C:/Users/DS/COS/최종/label 조합 변경')
sys.path.insert(0, str(OUTPUT_ROOT))
from epa_experiment_utils import run_endpoint, verify_endpoint
RUN_ID = os.environ.setdefault('EPA_RUN_ID', 'epa-production-20261106-no-catboost')
CSV_PATH = Path(r'C:/Users/DS/COS/최종/production데이터/skin_vectorized.csv')
{'endpoint':'skin','run_id':RUN_ID,'input':str(CSV_PATH),'output':str(OUTPUT_ROOT)}

{'endpoint': 'skin',
 'run_id': 'epa-production-20261106-no-catboost',
 'input': 'C:\\Users\\DS\\COS\\최종\\production데이터\\skin_vectorized.csv',
 'output': 'C:\\Users\\DS\\COS\\최종\\label 조합 변경'}

In [2]:
result = run_endpoint('skin', CSV_PATH, output_dir=OUTPUT_ROOT, run_id=RUN_ID)
verify_endpoint(result['output_dir'])

skin: 662 rows, 2265 features; artifacts=C:\Users\DS\COS\최종\label 조합 변경\runs\epa-production-20261106-no-catboost\skin\8d1059edb2caa233


DONE skin/A/lr outer=1/5 elapsed=0.8s


DONE skin/A/lr outer=2/5 elapsed=0.9s


DONE skin/A/lr outer=3/5 elapsed=0.9s


DONE skin/A/lr outer=4/5 elapsed=0.9s


DONE skin/A/lr outer=5/5 elapsed=0.9s


... (52 outputs left out)

DONE skin/B/lgbm outer=3/5 elapsed=1.1s


DONE skin/B/lgbm outer=4/5 elapsed=1.1s


DONE skin/B/lgbm outer=5/5 elapsed=1.1s


{'endpoint': 'skin',
 'identity': '8d1059edb2caa233b1d59c807134447e7069525b79c148fc38b8c49006c7dd9f',
 'combinations': 12,
 'outer_folds': 60,
 'status': 'passed'}

In [3]:
import pandas as pd
columns = ['endpoint', 'label', 'model', 'sensitivity', 'specificity', 'MCC', 'ROC-AUC',
           'abs_gap', 'PR-AUC', 'precision', 'F1', 'balanced_accuracy',
           'positive_count', 'negative_count', 'confusion_matrix',
           'sensitivity_std', 'specificity_std', 'MCC_std', 'ROC-AUC_std']
report = pd.DataFrame(result['report'])
for metric in ('sensitivity', 'specificity', 'MCC', 'ROC-AUC'):
    report[metric + '_std'] = report['fold_dispersion'].map(lambda values: values[metric])
report[columns]

,endpoint,label,model,sensitivity,specificity,MCC,ROC-AUC,abs_gap,PR-AUC,precision,F1,balanced_accuracy,positive_count,negative_count,confusion_matrix,sensitivity_std,specificity_std,MCC_std,ROC-AUC_std
0,skin,A,lr,0.513514,0.726400,0.121828,0.636130,0.212886,0.156860,0.100000,0.167401,0.619957,37,625,"[[454, 171], [18, 19]]",0.093984,0.083599,0.053901,0.080079
1,skin,A,rf,0.729730,0.675200,0.195351,0.755265,0.054530,0.218571,0.117391,0.202247,0.702465,37,625,"[[422, 203], [10, 27]]",0.136230,0.082482,0.038645,0.045088
2,skin,A,et,0.648649,0.668800,0.152992,0.689081,0.020151,0.188114,0.103896,0.179104,0.658724,37,625,"[[418, 207], [13, 24]]",0.096989,0.063649,0.038178,0.039639
3,skin,A,svm,0.702703,0.683200,0.187352,0.728022,0.019503,0.213278,0.116071,0.199234,0.692951,37,625,"[[427, 198], [11, 26]]",0.120923,0.030252,0.066266,0.074581
4,skin,A,xgb,0.675676,0.614400,0.135914,0.698573,0.061276,0.193713,0.093985,0.165017,0.645038,37,625,"[[384, 241], [12, 25]]",0.209165,0.064844,0.092596,0.075059
5,skin,A,lgbm,0.648649,0.662400,0.149321,0.683265,0.013751,0.204207,0.102128,0.176471,0.655524,37,625,"[[414, 211], [13, 24]]",0.096989,0.033178,0.044666,0.098424
6,skin,B,lr,0.631148,0.661111,0.232030,0.703294,0.029964,0.332980,0.296154,0.403141,0.646129,122,540,"[[357, 183], [45, 77]]",0.105698,0.088956,0.037107,0.025853
7,skin,B,rf,0.647541,0.688889,0.269711,0.727580,0.041348,0.380774,0.319838,0.428184,0.668215,122,540,"[[372, 168], [43, 79]]",0.056363,0.072198,0.073163,0.031362
8,skin,B,et,0.696721,0.672222,0.292520,0.725804,0.024499,0.367068,0.324427,0.442708,0.684472,122,540,"[[363, 177], [37, 85]]",0.119195,0.066319,0.076213,0.026992
9,skin,B,svm,0.647541,0.620370,0.209876,0.652444,0.027171,0.307236,0.278169,0.389163,0.633956,122,540,"[[335, 205], [43, 79]]",0.093101,0.065473,0.046923,0.026361


## 해석
PR-AUC는 Average Precision(AP)이며, 각 `_std` 열은 외부 5-fold의 표본 표준편차(ddof=1)입니다.
SVM의 fold별 margin은 공통 확률 척도로 보정되지 않았으므로 pooled ROC-AUC·PR-AUC는 기술적 순위 요약으로만 해석합니다.
sensitivity·specificity·MCC·ROC-AUC를 함께 확인합니다. sensitivity와 specificity의 절대 수준, 차이, fold 변동성으로 예측 균형을 판단하며 단일 지표 최고 모델을 선언하지 않습니다. 두 값이 모두 낮으면 차이가 작아도 좋은 결과가 아닙니다. A/B는 다른 예측 문제이며 SVM fold 간 margin은 보정된 확률이 아닙니다. 실제 fold별 임계값 예측을 합쳐 OOF 지표를 계산하며 전체 OOF 정답으로 임계값을 다시 선택하지 않습니다.